In [19]:
# Image-Based Classification: 3d cnns with batch normalization dropout and ReLU action
# Persistence Image Classification: 4-layer 2D CNNs trained for each of homological dimensions (3)
# Pre-classification layer encodings from 3 2D CNNs were combined with a multilayer percpetion, L1 regularization, and sigmoid activation 

In [6]:
import numpy as np
import pandas as pd
import pickle


In [37]:
# Merge labels with images
CLINICAL_PATH = "./clinical/ADNIMERGE_04May2025.csv"

clinical_df = pd.read_csv(CLINICAL_PATH)

# binary classification - alzheimer's disease or not 
# Diagnosis = DX DX_bl = diagnosis at baseline CN = cognitively normal, AD = Alzheimer's Disease 

# Remove records without a diagnosis nor image uid
clinical_df = clinical_df[~clinical_df['DX_bl'].isna()]
clinical_df = clinical_df[~clinical_df['IMAGEUID'].isna()]
# Remove dementia or non healthy controls of matched age to AD patients
print("Average age of AD patients: ", clinical_df['AGE'].mean())
print("Number of unique patients: ", clinical_df['RID'].nunique())
print(clinical_df['DX'].unique())
# only AD and CN patients
clinical_df = clinical_df[clinical_df['DX_bl'].isin(['CN', 'AD'])]
print("Number of AD patients: ", clinical_df[clinical_df['DX_bl'] == 'AD']['RID'].nunique())
print("Number of CN patients: ", clinical_df[clinical_df['DX_bl'] == 'CN']['RID'].nunique())
clinical_df[['PTID','RID', 'DX_bl', 'MMSE', 'IMAGEUID', 'Hippocampus', 'Ventricles', 'EXAMDATE']] # Diagnosis at baseline

Average age of AD patients:  73.15195690847993
Number of unique patients:  2360
['CN' 'Dementia' 'MCI' nan]
Number of AD patients:  406
Number of CN patients:  537


/tmp/ipykernel_3768459/2132680976.py:4: DtypeWarning: Columns (19,20,21,50,51,104,105,106) have mixed types. Specify dtype option on import or set low_memory=False.
  clinical_df = pd.read_csv(CLINICAL_PATH)


,PTID,RID,DX_bl,MMSE,IMAGEUID,Hippocampus,Ventricles,EXAMDATE
0,011_S_0002,2,CN,28.0,35475.0,8336.0,118233.0,2005-09-08
1,011_S_0003,3,AD,20.0,32237.0,5319.0,84599.0,2005-09-12
2,011_S_0003,3,AD,24.0,31863.0,5446.0,88580.0,2006-03-13
3,011_S_0003,3,AD,17.0,35576.0,5157.0,90099.0,2006-09-12
4,011_S_0003,3,AD,19.0,88252.0,5139.0,97420.0,2007-09-12
...,...,...,...,...,...,...,...,...
16145,941_S_6570,6570,CN,28.0,1621706.0,7394.3,32249.1,2022-09-20
16200,033_S_6352,6352,CN,30.0,1624603.0,8025.2,52448.2,2022-10-21
16207,168_S_6754,6754,AD,24.0,1638162.0,6053.9,53280.3,2022-10-25
16214,941_S_6574,6574,CN,30.0,1641654.0,6320.8,33670.5,2022-11-01


In [3]:
# COLLECT ALL PATIENT IDS 
import os
import re
PTID = set()
IMAGE_PATH = "./CROP_AND_SEGMENT"

# Regular expression pattern to extract the subject ID (002_S_0413)
pattern = re.compile(r"ADNI_(\d{3}_S_\d{4})_MR")

for filename in os.listdir(IMAGE_PATH):
    match = pattern.search(filename)
    if match:
        PTID.add(match.group(1))

PTID = sorted(PTID)
print(PTID)

['002_S_0413', '002_S_0559', '002_S_1018', '002_S_1070', '005_S_0324', '005_S_0553', '005_S_0572', '005_S_0814', '012_S_0689', '018_S_0335', '018_S_0369', '018_S_0450', '018_S_0633', '023_S_0030', '023_S_0031', '023_S_0058', '023_S_0061', '023_S_0331', '023_S_0376', '023_S_0388', '023_S_0604', '023_S_0625', '023_S_0916', '023_S_0926', '023_S_0963', '023_S_1046', '023_S_1262', '027_S_0307', '027_S_0403', '027_S_0404', '027_S_0835', '027_S_1081', '027_S_1082', '027_S_1385', '031_S_0830', '031_S_1066', '031_S_1209', '032_S_0677', '032_S_1169', '037_S_0303', '037_S_0501', '053_S_0507', '100_S_0015', '100_S_1286', '116_S_0382', '116_S_0487', '116_S_0649', '116_S_1249', '126_S_0606', '127_S_0260', '127_S_0622', '127_S_0844', '130_S_0956', '136_S_0086', '136_S_0184', '136_S_0195', '136_S_0196', '136_S_0300', '136_S_0426', '136_S_0429']


In [4]:
clinical_df_subset = clinical_df[clinical_df['PTID'].isin(PTID)]
clinical_df_subset['PTID'].nunique()
print("Number of AD patients: ", clinical_df_subset[clinical_df_subset['DX_bl'] == 'AD']['RID'].nunique())
print("Number of CN patients: ", clinical_df_subset[clinical_df_subset['DX_bl'] == 'CN']['RID'].nunique())

Number of AD patients:  18
Number of CN patients:  22


In [24]:

with open('None_CNNflexi_dense_patch217_run0CV_0_tdadim[0]_param3D.pkl', 'rb') as file: 
    data = pickle.load(file)
    print(data)

{'n_channels': 1, 'datadir': '/links/groups/borgwardt/Data/ADNI/DATA/3D/3Dpatches/data_patches_brain_extraction-complete_scaled_1e-6_999_innerfullIM120x144x120', 'patchid': 217, 'patchdim': (120, 144, 120), 'BNloc': 1, 'useDO': True, 'n_filters_1': 32, 'n_filters_2': 64, 'kernel_size': 4, 'l2': 0.0001, 'l1_den': nan, 'l1_act': 0, 'sample_shape': (10, 120, 144, 120, 1), 'clf': 'gap', 'n_layers': 5}


In [5]:
import tensorflow as tf
print("Num GPUs Available: ", len(tf.config.experimental.list_physical_devices('GPU')))

2025-05-07 14:03:42.620400: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-05-07 14:03:42.809206: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1746644622.877096 3765724 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1746644622.900145 3765724 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1746644623.053337 3765724 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

Num GPUs Available:  1


In [6]:
import torch
torch.cuda.empty_cache()

In [7]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import nibabel as nib
import numpy as np
import re
import torch
import torch.nn as nn
import torch.nn.functional as F
import os

In [38]:

#from tensorflow.python.keras.models import Sequential, Model
#from tensorflow.python.keras.layers import Dense, Flatten, Conv2D, MaxPooling2D, Dropout, BatchNormalization, Input, concatenate
#from tensorflow.python.keras.layers import Conv3D, MaxPooling3D, Activation, GlobalAveragePooling2D,GlobalAveragePooling3D

# -----------------------------
# Custom Dataset
# -----------------------------
class AlzheimerDataset(Dataset):
    def __init__(self, data_dir, patient_diag_df, use_tda=False):
        self.data_dir = data_dir
        self.use_tda = use_tda
        self.samples = []
        IMAGE_PATH = "./CROP_AND_SEGMENT"

        # Regular expression pattern to extract the subject ID (002_S_0413)
        pattern = re.compile(r"ADNI_(\d{3}_S_\d{4})_MR")
        diag_dict = dict(zip(patient_diag_df['PTID'], patient_diag_df['DX_bl']))
        for filename in os.listdir(IMAGE_PATH):
            match = pattern.search(filename)
            if match:
                ptid = match.group(1)
                if ptid in diag_dict:
                    diagnosis = diag_dict[ptid]
                    self.samples.append((filename, ptid, diagnosis))
        # Map textual labels to integers
        self.label_map = {"CN": 0, "AD": 1}     
        

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        filename, ptid, diagnosis = self.samples[idx]

        # Load MRI
        mri_path = os.path.join(self.data_dir, filename)
        mri_img = nib.load(mri_path).get_fdata()
        mri_tensor = torch.tensor(mri_img, dtype=torch.float32).unsqueeze(0)  # [1, D, H, W]

        # Get Labels
        label = self.label_map.get(diagnosis, -1)
        label_tensor = torch.tensor(label, dtype=torch.long)
        # Load TDA
        if self.use_tda:
            # Dummy TDA features
            tda0 = torch.rand(10)
            tda1 = torch.rand(10)
            tda2 = torch.rand(10)
            return mri_tensor, tda0, tda1, tda2, label_tensor
        else:
            return mri_tensor, label_tensor

        #return torch.tensor(mri), torch.tensor(tda[0]), torch.tensor(tda[1]), torch.tensor(tda[2]), torch.tensor(label)
# Model Definitions 

class TDABranch(nn.Module):
    def __init__(self):
        super(TDABranch, self).__init__()
        self.conv1 = nn.Conv2d(1, 16, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.fc = nn.Linear(32 * 12 * 12, 128)  # if input is 50x50 and two pool layers

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc(x))
        return x
class MRI3DCNN(nn.Module):
    def calculate_flattened_size(self, input_shape):
       with torch.no_grad():
            dummy = torch.zeros(1, 1, *input_shape)  # Batch size 1, channels 1
            out = self._forward_features(dummy)
            return out.view(1, -1).size(1)
    def __init__(self, input_channels =1 ): # grayscale single volume 
        num_filters = 8# paper- 32 used fewer because its crashing 
        self.input_shape = (120, 144, 120)
        self.n_classes = 2 # binary classification
        super(MRI3DCNN, self).__init__()
        self.conv1 = nn.Conv3d(in_channels=input_channels, out_channels=num_filters, kernel_size=4, padding=1, bias=True)
        self.bn = nn.BatchNorm3d(num_filters, affine=True)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool3d(kernel_size=2)
        self.dropout = nn.Dropout3d(0.25)
        #self.conv2 = nn.Conv3d(8, 16, kernel_size=3, padding=1)
        #self.fc = nn.Linear(16 * 16 * 16 * 16, 128)  # Adjust for your image shape

        # Compute flattened size after conv+pool for fully connected layers
        self.flattened_size = self.calculate_flattened_size(self.input_shape)

        self.fc1 = nn.Linear(self.flattened_size, 32)
        #self.fc2 = nn.Linear(256, 128)
        #self.fc3 = nn.Linear(256, 64)
        #self.fc2 = nn.Linear(128, 32)
        # classify as AD or CN (2 classes)
        self.classifier = nn.Sequential(
            # nn.Linear(64, 32),
            # nn.ReLU(),
            nn.Linear(32, self.n_classes),
            nn.Sigmoid()
        )

    def _forward_features(self, x):
        x = self.relu(self.bn(self.conv1(x)))
        x = self.pool(x)
        x = self.dropout(x)
        return x

    def forward(self, x):
        x = self._forward_features(x)
        x = x.view(x.size(0), -1)  # Flatten
        x = F.relu(self.fc1(x))
        #x = F.relu(self.fc2(x))
        #x = F.relu(self.fc3(x))
        x = self.classifier(x)
        return x
    
class CombinedModel(nn.Module):
    def __init__(self):
        super(CombinedModel, self).__init__()
        self.mri_branch = MRI3DCNN()
        self.tda_0 = TDABranch()
        self.tda_1 = TDABranch()
        self.tda_2 = TDABranch()
        
        self.classifier = nn.Sequential(
            nn.Linear(128 * 4, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 2)  # 2 classes: AD / CN
        )

    def forward(self, mri, tda0, tda1, tda2):
        mri_out = self.mri_branch(mri)
        tda0_out = self.tda_0(tda0)
        tda1_out = self.tda_1(tda1)
        tda2_out = self.tda_2(tda2)
        combined = torch.cat([mri_out, tda0_out, tda1_out, tda2_out], dim=1)
        return self.classifier(combined)
# -----------------------------
# Model Setup
# -----------------------------
# Assume CombinedModel is already defined from previous message

# -----------------------------
# Training Function
# -----------------------------
def train_model(model, dataloader, num_epochs=10, lr=1e-4, use_tda=False):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    torch.cuda.empty_cache()
    print(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        batch = 0
        if use_tda: 
            for mri, tda0, tda1, tda2, label in dataloader:
                mri, tda0, tda1, tda2, label = mri.to(device), tda0.to(device), tda1.to(device), tda2.to(device), label.to(device)

                optimizer.zero_grad()
                outputs = model(mri, tda0, tda1, tda2)
                loss = criterion(outputs, label)
                loss.backward()
                optimizer.step()

                running_loss += loss.item()
                _, predicted = torch.max(outputs.data, 1)
                total += label.size(0)
                correct += (predicted == label).sum().item()

                print(f"Epoch {epoch+1}/{num_epochs}, Batch {batch}, Loss: {running_loss:.4f}, Accuracy: {100 * correct / total:.2f}%")
                batch += 1
        else: 
            for mri, label in dataloader:
                mri, label = mri.to(device), label.to(device)

                optimizer.zero_grad()
                outputs = model(mri)
                loss = criterion(outputs, label)
                loss.backward()
                optimizer.step()

                running_loss += loss.item()
                _, predicted = torch.max(outputs.data, 1)
                total += label.size(0)
                correct += (predicted == label).sum().item()
                batch += 1
                print(f"Epoch {epoch+1}/{num_epochs},  Batch {batch}, Loss: {running_loss:.4f}, Accuracy: {100 * correct / total:.2f}%")
    print("Training complete")
    return model

# -----------------------------
# Execution
# -----------------------------
data_path = './CROP_AND_SEGMENT'  # Replace with actual path
dataset = AlzheimerDataset(data_path, clinical_df)
# 2. Split into train, val, and test (e.g. 70/15/15)
total_size = len(dataset)
train_size = int(0.7 * total_size)
val_size = 0#int(0.15 * total_size)
test_size = total_size - train_size - val_size
print("Train size: ", train_size, " Val Size: ", val_size, " Test Size: ", test_size)
train_set, val_set, test_set = random_split(dataset, [train_size, val_size, test_size])

# 3. DataLoaders
train_loader = DataLoader(train_set, batch_size=3, shuffle=True)
val_loader = DataLoader(val_set, batch_size=3, shuffle=False)
img_test_loader = DataLoader(test_set, batch_size=3, shuffle=False)

#dataloader = DataLoader(dataset, batch_size=4, shuffle=True)

model = MRI3DCNN()
#trained_model = train_model(model, train_loader, num_epochs=10)




Train size:  144  Val Size:  0  Test Size:  62


In [ ]:
torch.save(trained_model.state_dict(), "cnn3d_images_trained_model.pth")

print("PyTorch model saved successfully!")

PyTorch model saved successfully!


In [30]:
with open("cnn3d_images_trained_model.pkl", "wb") as f:
    pickle.dump(trained_model, f)

print("Model saved successfully!")

Model saved successfully!


In [33]:
import nibabel as nib
class TDADataset(Dataset):
    def __init__(self, data_dir, patient_diag_df, dimension):
        # data_dir (str): Path to TDA_V2 directory containing persistence images.
        #  patient_diag_df (DataFrame): DataFrame mapping PTID to AD/CN diagnosis.
        # dimension (int): Dimension (0, 1, or 2) for persistence images.
        self.data_dir = data_dir
        self.dimension = dimension
        self.samples = []
        
        # Regular expression to extract subject ID (e.g., "002_S_0413") from filenames
        pattern = re.compile(r"ADNI_(\d{3}_S_\d{4})_MR")
        dim_pattern = re.compile(r"_tda_(\d)\.npy$")
        diag_dict = dict(zip(patient_diag_df['PTID'], patient_diag_df['DX_bl']))
        
        for filename in os.listdir(data_dir):
            if dim_pattern.search(filename):  # Ensuring correct dimension
                match = pattern.search(filename)
                if match:
                    ptid = match.group(1)
                    if ptid in diag_dict:
                        diagnosis = diag_dict[ptid]
                        self.samples.append((os.path.join(data_dir, filename), ptid, diagnosis))

        # Map textual labels to integers (CN=0, AD=1)
        self.label_map = {"CN": 0, "AD": 1}

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        file_path, ptid, diagnosis = self.samples[idx]
        # Load NIfTI image
        nii_img = np.load(file_path) #.get_fdata()

        # Resize if needed (adjust based on desired shape)
        nii_img = torch.tensor(nii_img, dtype=torch.float32).unsqueeze(0)  # [1, D, H, W]

        # Get label
        label = self.label_map.get(diagnosis, -1)
        label_tensor = torch.tensor(label, dtype=torch.long)

        return nii_img, label_tensor
# CNN Model for TDA Persistence Images per Dimension 
class TDADimCNN(nn.Module):
    def __init__(self):
        # INPUT SIZE: 50 X 50
        super(TDADimCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 16, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.fc = nn.Linear(32 * 12 * 12, 128)  # if input is 50x50 and two pool layers
        self.classifier = nn.Linear(128, 2)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc(x))
        return self.classifier(x)
    
def train_tda_model(model, dataloader, num_epochs=10, lr=1e-4, save_path="trained_model.pth"):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {running_loss:.4f}, Accuracy: {100 * correct / total:.2f}%")

    return model 

numDims = 3
# set seed so dataloader is reproducible for testing 
torch.manual_seed(12) 

for dim in range(numDims):
    data_path = './TDA_V2'  # Replace with actual path
    dataset = TDADataset(data_path, clinical_df, dim)
    # 2. Split into train, val, and test (e.g. 70/15/15)
    total_size = len(dataset)
    train_size = int(0.7 * total_size)
    val_size = 0#int(0.15 * total_size)
    test_size = total_size - train_size - val_size
    print("Train size: ", train_size, " Val Size: ", val_size, " Test Size: ", test_size)
    tda_train_set, tda_val_set, tda_test_set = random_split(dataset, [train_size, val_size, test_size])

    # 3. DataLoaders
    tda_train_loader = DataLoader(tda_train_set, batch_size=4, shuffle=True)
    tda_val_loader = DataLoader(tda_val_set, batch_size=4, shuffle=False)
    tda_test_loader = DataLoader(tda_test_set, batch_size=4, shuffle=False)

    tdamodel = TDADimCNN()
    # TDA_trained_model = train_tda_model(tdamodel, tda_train_loader, num_epochs=40)

    # # save models!
    # torch.save(TDA_trained_model.state_dict(), "tda_"+ str(dim) +"_trained_model.pth")
    # print("PyTorch model saved successfully!")
    # with open("tda_"+ str(dim) +"_trained_model.pkl", "wb") as f:
    #     pickle.dump(trained_model, f)
    # print("Model saved successfully!")


Train size:  180  Val Size:  0  Test Size:  78
Train size:  180  Val Size:  0  Test Size:  78
Train size:  180  Val Size:  0  Test Size:  78


In [34]:
# do a CNN with all 3 dimensions of persistence images 
import nibabel as nib
class TDA3DDataset(Dataset):
    def __init__(self, data_dir, patient_diag_df):

        # data_dir (str): Path to TDA_V2 directory containing persistence images.
        #  patient_diag_df (DataFrame): DataFrame mapping PTID to AD/CN diagnosis.
        # dimension (int): Dimension (0, 1, or 2) for persistence images.
        self.data_dir = data_dir
        self.samples = []

        # Regular expression to extract subject ID (e.g., "002_S_0413") from filenames
        pattern = re.compile(r"ADNI_(\d{3}_S_\d{4})_MR")
        diag_dict = dict(zip(patient_diag_df['PTID'], patient_diag_df['DX_bl']))

        # Organize files by subject ID
        file_dict = {}  
        for filename in os.listdir(data_dir):
            match = pattern.search(filename)
            if match:
                ptid = match.group(1)
                if ptid not in file_dict:
                    file_dict[ptid] = []
                file_dict[ptid].append(os.path.join(data_dir, filename))

        # Filter to ensure only patients with all three dimensions are included
        for ptid, paths in file_dict.items():
            if ptid in diag_dict:
                if all(any(f"_tda_{dim}.npy" in p for p in paths) for dim in range(3)):
                    sorted_paths = [next(p for p in paths if f"_tda_{dim}.npy" in p) for dim in range(3)]
                    diagnosis = diag_dict[ptid]
                    self.samples.append((sorted_paths, ptid, diagnosis))

        # Map textual labels to integers (CN=0, AD=1)
        self.label_map = {"CN": 0, "AD": 1}

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        file_paths, ptid, diagnosis = self.samples[idx]

        # Load all three persistence images and stack them
        tda_images = [torch.tensor(np.load(p), dtype=torch.float32).unsqueeze(0) for p in file_paths]  # [1, 50, 50]

        # Stack into a tensor with 3 channels
        tda_tensor = torch.cat(tda_images, dim=0)  # [3, 50, 50] to match CNN expectation

        # Get label
        label = self.label_map.get(diagnosis, -1)
        label_tensor = torch.tensor(label, dtype=torch.long)
        #print(tda_tensor.shape)
        return tda_tensor, label_tensor
# CNN Model for TDA Persistence Images per Dimension 
class TDA3DCNN(nn.Module):
    def __init__(self):
        super(TDA3DCNN, self).__init__()
        # INPUT SIZE = 3 X (50 * 50) (3 channels)
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1)  # Input has 3 channels
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.fc = nn.Linear(32 * 12 * 12, 128)  # Adjust based on input shape
        self.classifier = nn.Linear(128, 2)  # Binary classification (AD/CN)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)  # Flatten
        x = F.relu(self.fc(x))
        return self.classifier(x)
    
def train_tda_model(model, dataloader, num_epochs=10, lr=1e-4, save_path="trained_model.pth"):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {running_loss:.4f}, Accuracy: {100 * correct / total:.2f}%")

    return model 

numDims = 1
# set seed so dataloader is reproducible for testing 
torch.manual_seed(12) 

for dim in range(numDims):
    data_path = './TDA_V2'  # Replace with actual path
    dataset = TDA3DDataset(data_path, clinical_df)
    # 2. Split into train, val, and test (e.g. 70/15/15)
    total_size = len(dataset)
    train_size = int(0.7 * total_size)
    val_size = 0#int(0.15 * total_size)
    test_size = total_size - train_size - val_size
    print("Train size: ", train_size, " Val Size: ", val_size, " Test Size: ", test_size)
    tda_train_set, tda_val_set, tda_test_set = random_split(dataset, [train_size, val_size, test_size])

    # 3. DataLoaders
    tda_train_loader = DataLoader(tda_train_set, batch_size=4, shuffle=True)
    tda_val_loader = DataLoader(tda_val_set, batch_size=4, shuffle=False)
    tda3d_test_loader = DataLoader(tda_test_set, batch_size=4, shuffle=False)

    sample_image, sample_label = dataset[0]
    
    print("Sample Image Shape:", sample_image.shape)  # Should be [3, 50, 50]
    print("Sample Label:", sample_label)

    tdamodel = TDA3DCNN()
    TDA_trained_model = train_tda_model(tdamodel, tda_train_loader, num_epochs=40)

    # save models!
    torch.save(TDA_trained_model.state_dict(), "tda_3d_trained_model.pth")
    print("PyTorch model saved successfully!")
    #with open("tda_3d_trained_model.pkl", "wb") as f:
        #pickle.dump(trained_model, f)
    #print("Model saved successfully!")


Train size:  26  Val Size:  0  Test Size:  12
Sample Image Shape: torch.Size([3, 50, 50])
Sample Label: tensor(1)
Epoch 1/40, Loss: 27.5121, Accuracy: 65.38%
Epoch 2/40, Loss: 10.2678, Accuracy: 38.46%
Epoch 3/40, Loss: 12.3765, Accuracy: 42.31%
Epoch 4/40, Loss: 15.7684, Accuracy: 53.85%
Epoch 5/40, Loss: 26.0063, Accuracy: 46.15%
Epoch 6/40, Loss: 10.0926, Accuracy: 50.00%
Epoch 7/40, Loss: 4.6788, Accuracy: 57.69%
Epoch 8/40, Loss: 5.5284, Accuracy: 57.69%
Epoch 9/40, Loss: 4.7772, Accuracy: 69.23%
Epoch 10/40, Loss: 6.6116, Accuracy: 57.69%
Epoch 11/40, Loss: 4.0537, Accuracy: 76.92%
Epoch 12/40, Loss: 6.2251, Accuracy: 57.69%
Epoch 13/40, Loss: 5.4586, Accuracy: 69.23%
Epoch 14/40, Loss: 10.7705, Accuracy: 46.15%
Epoch 15/40, Loss: 11.8849, Accuracy: 46.15%
Epoch 16/40, Loss: 15.9458, Accuracy: 50.00%
Epoch 17/40, Loss: 6.1735, Accuracy: 53.85%
Epoch 18/40, Loss: 4.3336, Accuracy: 65.38%
Epoch 19/40, Loss: 3.9536, Accuracy: 69.23%
Epoch 20/40, Loss: 6.0194, Accuracy: 69.23%
Epoch 

In [35]:
# Model that uses preprocessed and segmented (hippocampus) MRI and TDAs 
class TDA_MRI_Dataset(Dataset):
    def __init__(self, tda_dir, mri_dir, patient_diag_df):

        
        #    tda_dir (str): Path to directory containing persistence images (.npy).
        # mri_dir (str): Path to directory containing MRI scans (.nii.gz).
        # patient_diag_df (DataFrame): DataFrame mapping PTID to AD/CN diagnosis.
 
        self.tda_dir = tda_dir
        self.mri_dir = mri_dir
        self.samples = []

        # Regular expression to extract patient ID (e.g., "002_S_0413")
        pattern = re.compile(r"ADNI_(\d{3}_S_\d{4})_MR")
        diag_dict = dict(zip(patient_diag_df['PTID'], patient_diag_df['DX_bl']))

        # Organize files by subject ID
        file_dict = {}  
        for filename in os.listdir(tda_dir):
            match = pattern.search(filename)
            if match:
                ptid = match.group(1)
                if ptid not in file_dict:
                    file_dict[ptid] = []
                file_dict[ptid].append(os.path.join(tda_dir, filename))

        # Index MRI files dynamically
        mri_dict = {re.search(pattern, f).group(1): os.path.join(mri_dir, f) for f in os.listdir(mri_dir) if re.search(pattern, f)}

        # Ensure we have all 3 TDA images and the corresponding MRI
        for ptid, paths in file_dict.items():
            if ptid in diag_dict and ptid in mri_dict:
                if all(any(f"_tda_{dim}.npy" in p for p in paths) for dim in range(3)):
                    sorted_paths = [next(p for p in paths if f"_tda_{dim}.npy" in p) for dim in range(3)]
                    mri_path = mri_dict[ptid]  # Retrieve matching MRI file dynamically

                    diagnosis = diag_dict[ptid]
                    self.samples.append((sorted_paths, mri_path, ptid, diagnosis))

        # Map textual labels to integers (CN=0, AD=1)
        self.label_map = {"CN": 0, "AD": 1}

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        file_paths, mri_path, ptid, diagnosis = self.samples[idx]

        # Load all three persistence images correctly
        tda_images = [torch.tensor(np.load(p), dtype=torch.float32).unsqueeze(0) for p in file_paths]  # [1, 50, 50]
        tda_tensor = torch.cat(tda_images, dim=0)  # [3, 50, 50] for CNN input

        # Load MRI volume
        mri_img = nib.load(mri_path).get_fdata()
        mri_tensor = torch.tensor(mri_img, dtype=torch.float32).unsqueeze(0)  # [1, D, H, W]
        #mri_tensor = mri_tensor.unsqueeze(0)
        # Get label
        label = self.label_map.get(diagnosis, -1)
        label_tensor = torch.tensor(label, dtype=torch.long)
        #print("Raw MRI Shape:", mri_img.shape)  # Expected: (D, H, W)
        #print("MRI Tensor Shape After Unsqueeze:", mri_tensor.shape) 
        return tda_tensor, mri_tensor, label_tensor
    
class TDA_MRI_Model(nn.Module):
    def calculate_tda_flattened_size(self, input_shape):
        with torch.no_grad():
            dummy = torch.zeros(1, *input_shape)
            out = self.tda_forward_features(dummy)
            return out.view(1, -1).size(1)
    def tda_forward_features(self, x):
       # Extracts feature maps before final classification.
        x = self.tda_pool(self.tda_conv1(x))
        x = self.tda_pool(self.tda_conv2(x))
        return x
    def calculate_mri_flattened_size(self, input_shape):
            with torch.no_grad():
                dummy = torch.zeros(1, *input_shape)
                out = self.mri_forward_features(dummy)
                return out.view(1, -1).size(1)
    def mri_forward_features(self, x):
        #Extracts feature maps before final classification.
        x = self.mri_pool(self.mri_conv1(x))
        return x
    def __init__(self):
        super(TDA_MRI_Model, self).__init__()
        self.mri_input_shape = (120, 144, 120)
        self.tda_input_shape = (3, 50, 50)
        # Persistence Image Branch (2D CNN)
        self.tda_conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)  # 3-channel TDA input
        self.tda_pool = nn.MaxPool2d(2, 2)
        self.tda_conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.flattened_tda_size = self.calculate_tda_flattened_size(self.tda_input_shape)
        #print(self.flattened_tda_size)
        self.tda_fc = nn.Linear(self.flattened_tda_size, 64)  # Flattened feature

        # MRI Volume Branch (3D CNN)
        self.mri_conv1 = nn.Conv3d(1, 8, kernel_size=4, padding=1)
        self.mri_pool = nn.MaxPool3d(2)
        self.flattened_mri_size = self.calculate_mri_flattened_size(self.mri_input_shape)
        #print(self.flattened_mri_size)
        self.mri_fc = nn.Linear(self.flattened_mri_size, 64) 
        self.final_fc = nn.Sequential(
            nn.Linear(128, 2),
            nn.Sigmoid()  # 2 classes: AD / CN
        )

    def forward(self, tda, mri):
        # Persistence Image Processing
        tda = self.tda_pool(F.relu(self.tda_conv1(tda)))
        tda = self.tda_pool(F.relu(self.tda_conv2(tda)))
        tda = tda.view(tda.size(0), -1)  # Flatten
        tda = F.relu(self.tda_fc(tda))

        # MRI Processing
        mri = self.mri_pool(F.relu(self.mri_conv1(mri)))
        mri = mri.view(mri.size(0), -1)  # Flatten
        mri = F.relu(self.mri_fc(mri))

        # Combine Features
        combined = torch.cat([tda, mri], dim=1)
        return self.final_fc(combined)
    
# Training function
def train_model(model, dataloader, num_epochs=10, lr=1e-4, save_path="trained_tda_mri_model.pth"):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = TDA_MRI_Model().to(device)
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        for tda, mri, labels in dataloader:
            tda, mri, labels = tda.to(device), mri.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(tda, mri)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {running_loss:.4f}, Accuracy: {100 * correct / total:.2f}%")
    return model


data_path_tda = "./TDA_V2"
data_path_mri = "./CROP_AND_SEGMENT"
dataset = TDA_MRI_Dataset(data_path_tda, data_path_mri, clinical_df)
tda_sample, mri_sample, label_sample = dataset[0]  

print("TDA Sample Shape:", tda_sample.shape)  # Expected: [3, 50, 50]
print("MRI Sample Shape:", mri_sample.shape)  # Expected: [1, D, H, W]
print("Label:", label_sample)
# Ensure reproducibility
torch.manual_seed(42)

# Split dataset (Train: 70%, Val: 15%, Test: 15%)
train_size = int(0.7 * len(dataset))
val_size = 0 #int(0.15 * len(dataset))
test_size = len(dataset) - train_size - val_size
train_set, val_set, test_set = random_split(dataset, [train_size, val_size, test_size])

# Dataloader setup
batch_size = 1
train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False)
combotest_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False)
model = TDA_MRI_Model()
# combo_trained_model = train_model(model, train_loader, num_epochs=10)
# save_path = './combo_trained_model.pth'
# # Save trained model
# torch.save(model.state_dict(), save_path)
# print(f"Model saved to {save_path}")
# with open("combo_trained_model.pkl", "wb") as f:
#     pickle.dump(combo_trained_model, f)
#     print("Model saved successfully!")


TDA Sample Shape: torch.Size([3, 50, 50])
MRI Sample Shape: torch.Size([1, 120, 144, 120])
Label: tensor(1)


In [46]:

from sklearn.metrics import accuracy_score, roc_auc_score, average_precision_score, recall_score, precision_score
def evaluate_model(model, test_loader, device):
    model.to(device)
    model.eval()

    all_labels = []
    all_preds = []
    all_probs = []  # Store probabilities for AUC/AP calculations

    with torch.no_grad():
        if isinstance(model, TDADimCNN) or isinstance(model, TDA3DCNN) :
            for tda, labels in test_loader:
                tda, labels = tda.to(device), labels.to(device)

                #if isinstance(model, TDADimCNN):  # If it's a 2D CNN, ignore MRI
                outputs = model(tda)

                probs = F.softmax(outputs, dim=1)[:, 1].cpu().numpy()  # Get probability for positive class
                preds = torch.argmax(outputs, dim=1).cpu().numpy()
                if isinstance(model, TDADimCNN):
                    threshold = 0.35  # Example: Lower threshold to increase positive predictions
                    preds = (probs > threshold).astype(int)
                else:
                    threshold = 0.005
                    preds = (probs < threshold).astype(int)
                labels = labels.cpu().numpy()
                all_labels.extend(labels)
                all_preds.extend(preds)
                all_probs.extend(probs)
        elif isinstance(model, MRI3DCNN):
            for mri, labels in test_loader:
                mri, labels = mri.to(device), labels.to(device)
                if isinstance(model, MRI3DCNN):  # If it's a 3D CNN, ignore TDA
                    outputs = model(mri)

                probs = F.softmax(outputs, dim=1)[:, 1].cpu().numpy()  # Get probability for positive class
                preds = torch.argmax(outputs, dim=1).cpu().numpy()
                #threshold = 0.3  # Example: Lower threshold to increase positive predictions
                #preds = (probs > threshold).astype(int)
                labels = labels.cpu().numpy()
                all_labels.extend(labels)
                all_preds.extend(preds)
                all_probs.extend(probs)
        else:
            for tda, mri, labels in test_loader:
                tda, mri, labels = tda.to(device), mri.to(device), labels.to(device)


                outputs = model(tda, mri)  # If it's MRI+TDA (3D CNN) 

                probs = F.softmax(outputs, dim=1)[:, 1].cpu().numpy()  # Get probability for positive class
                
                threshold = 0.5  # Example: Lower threshold to increase positive predictions
                preds = (probs > threshold).astype(int)
                
                #preds = torch.argmax(outputs, dim=1).cpu().numpy()
                labels = labels.cpu().numpy()
                print(labels)
                all_labels.extend(labels)
                all_preds.extend(preds)
                all_probs.extend(probs)
    # Compute metrics
    acc = accuracy_score(all_labels, all_preds)
    auc = roc_auc_score(all_labels, all_probs)
    aps = average_precision_score(all_labels, all_probs)
    recall = recall_score(all_labels, all_preds, zero_division=1)
    precision = precision_score(all_labels, all_preds, zero_division=1)
    print(all_probs)
    print(all_labels)
    results = {
        "Accuracy": acc,
        "AUC": auc,
        "Average Precision Score": aps,
        "Recall": recall,
        "Precision": precision
    }

    return results

# Load test dataset
#test_loader = DataLoader(test_set, batch_size=1, shuffle=False)

# Define models (assuming you have classes: MRI_CNN, TDA_CNN, TDA3D_CNN, MRI_TDA_CNN)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load trained models
models = {
    "MRI_3D_CNN": MRI3DCNN(),
    "TDA_2D_CNN_Dim0": TDADimCNN(),
    "TDA_2D_CNN_Dim1": TDADimCNN(),
    "TDA_2D_CNN_Dim2": TDADimCNN(),
    "TDA_3D_CNN": TDA3DCNN(),
    "MRI+TDA_3D_CNN": TDA_MRI_Model()
}

# Load saved weights
model_paths = {
    "MRI_3D_CNN": "./trained_model.pth",
    "TDA_2D_CNN_Dim0": "./tda_0_trained_model.pth",
    "TDA_2D_CNN_Dim1": "./tda_1_trained_model.pth",
    "TDA_2D_CNN_Dim2": "./tda_2_trained_model.pth",
    "TDA_3D_CNN": "./tda_3d_trained_model.pth",
    "MRI+TDA_3D_CNN": "./combo_trained_model.pth"
}
test_loaders = {
    "MRI_3D_CNN": img_test_loader,
    "TDA_2D_CNN_Dim0": tda_test_loader,
    "TDA_2D_CNN_Dim1": tda_test_loader,
    "TDA_2D_CNN_Dim2": tda_test_loader,
    "TDA_3D_CNN": tda3d_test_loader,
    "MRI+TDA_3D_CNN": combotest_loader
}

# Evaluate each model and print results
for model_name, model in models.items():
    # make new test_loader or get the old ones
    #test_loader = DataLoader(test_set, batch_size=1, shuffle=False)

    model.load_state_dict(torch.load(model_paths[model_name], map_location=device, weights_only=True))
    results = evaluate_model(model, test_loaders[model_name], device)
    
    print(f"\n**Results for {model_name}**")
    for metric, value in results.items():
        print(f"{metric}: {value:.4f}")

[0.26894143, 0.26894143, 0.26894143, 0.26894143, 0.26894143, 0.26894143, 0.26894143, 0.26894143, 0.26894143, 0.26894143, 0.26894143, 0.26894143, 0.26894143, 0.26894143, 0.26894143, 0.26894143, 0.26894143, 0.26894143, 0.26894143, 0.26894143, 0.26894143, 0.26894143, 0.26894143, 0.26894143, 0.26894143, 0.26894143, 0.26894143, 0.26894143, 0.26894143, 0.26894143, 0.26894143, 0.26894143, 0.26894143, 0.26894143, 0.26894143, 0.26894143, 0.26894143, 0.26894143, 0.26894143, 0.26894143, 0.26894143, 0.26894143, 0.26894143, 0.26894143, 0.26894143, 0.26894143, 0.26894143, 0.26894143, 0.26894143, 0.26894143, 0.26894143, 0.26894143, 0.26894143, 0.26894143, 0.26894143, 0.26894143, 0.26894143, 0.26894143, 0.26894143, 0.26894143, 0.26894143, 0.26894143]
[0, 0, 1, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 1, 0, 0, 0, 0, 1]

**Results for MRI_3D_CNN**
Accuracy: 0.6129
AUC: 0.5000
Average Pre